#### 1. Основные функции в координации e-commerce

Аналитик обеспечивает связность процессов, предоставляя единую версию правды (Single Source of Truth) для всех отделов:

*   **Мониторинг KPI в реальном времени:** Аналитик создает и поддерживает дашборды (например, в Apache Superset), отслеживая ключевые показатели: выручка, конверсия (CR), 
средний чек (AOV), стоимость привлечения клиента (CAC), пожизненная ценность (LTV).

*   **Координация маркетинга и продаж:** Аналитик выявляет, какие каналы приносят самую высокую прибыль, и рекомендуют перераспределять бюджет. Он анализирует эффективность акций и промокодов, координируя действия отдела продаж с маркетингом.

*   **Оптимизация воронки продаж (CRO):** Аналитик находит узкие места в пути пользователя (где происходит отказ от корзины, где долгая загрузка) и координирует работу с дизайнерами и разработчиками для улучшения UX/UI.

*   **Синхронизация с логистикой и складом:** Аналитик прогнозирует спрос на основе исторических данных, помогая избежать дефицита (out-of-stock) или затоваривания.

#### 2. Практические сценарии (Co-working)

**Маркетинг** Анализ ROAS (возврат расходов на рекламу), когортный анализ, эффективность e-mail рассылок.

**Product/UX** A/B тестирование новых фич сайта, анализ поведения пользователей (тепловые карты, пути).

**Категорийный менеджмент** ABC/XYZ-анализ ассортимента, поиск товаров с высокой маржой, анализ ценообразования конкурентов.

**Логистика** Анализ скорости доставки, точности комплектации заказов (order accuracy).

#### 3. Ценность для бизнеса

Аналитик данных в роли координатора позволяет:

*   **Принимать обоснованные решения:** Переход от интуитивного управления к управлению на основе данных.

*   **Быстро реагировать:** Мгновенно  замечать аномалии (например, резкое падение конверсии) и координировать их устранение.

*   **Персонализировать опыт:** Сегментировать клиентов для повышения повторных продаж.

В итоге, аналитик данных становится **"мозговым центром"**, который координирует все части e-commerce механизма для максимизации прибыли.

### Работа аналитика в интернет-магазине электронной техники

Жизненный цикл гаджетов короткий, а цена ошибки (затоваривание склада устаревшими моделями или дефицит новинок) крайне высока. 

#### 1. Сбор и подготовка данных (PostgreSQL)

Первым шагом аналитик извлекает исторические данные о продажах, остатках и поступлениях. В электронике важно учитывать не только количество, но и категории (напримеп, "Смартфоны" продаются быстрее, чем "Холодильники").
**Пример  SQL-запроса для выгрузки агрегатных данных:**

In [ ]:
# Настройка путей и импортов
import sys
from pathlib import Path

# Найдём корень проекта (где лежит README.md или config.py)
def find_project_root(marker="config.py"):
    current = Path().resolve()
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise RuntimeError(f"Project root with '{marker}' not found!")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

# Добавим корень проекта в sys.path, чтобы импортировать src и config
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Путь к данным
CSV_PATH = PROJECT_ROOT / "data" / "raw" / "report.csv"
assert CSV_PATH.exists(), f"CSV file not found at {CSV_PATH}"
print(f"CSV file: {CSV_PATH}")

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from config import RAW_DATA_DIR
from src.db.queries import run_query


In [8]:
# Автоматически перезагружает модули при изменении кода в .py-файлах.
# Не нужно перезапускать ядро после каждого изменения в src/.

# %load_ext autoreload
# %autoreload 2

In [4]:
df = pd.read_csv(
    CSV_PATH,                      # путь к файлу
    sep=';',                       # разделитель - точка с запятой
    parse_dates=['sale_date'],     # колонка для преобразования в дату
    encoding='utf-8',              # кодировка файла
    dtype={'product_id': 'int64'}  # тип данных для колонки product_id
                                   #  nrows=100 (прочитать только первые 100 строк)
)

In [16]:
# Сортируем от новых к старым
df_sorted = df.sort_values('sale_date', ascending=False)

# Проверяем результат
display(df_sorted.head(10))

,product_id,category,model_name,sale_date,quantity,revenue,stock_on_hand
49083,5,Clothing,Model_5,2026-03-07,3,97.50,47
49031,3,Clothing,Model_3,2026-03-06,6,640.44,110
49082,7,Clothing,Model_7,2026-03-06,3,181.35,109
49081,15,Clothing,Model_15,2026-03-06,4,234.80,21
49080,13,Clothing,Model_13,2026-03-06,4,216.56,25
49079,5,Clothing,Model_5,2026-03-06,4,130.00,23
49078,9,Clothing,Model_9,2026-03-06,4,133.60,103
49077,9,Clothing,Model_9,2026-03-06,2,66.80,103
49076,13,Clothing,Model_13,2026-03-06,6,324.84,25
49075,16,Electronics,Model_16,2026-03-06,6,265.08,46


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49084 entries, 49083 to 0
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   product_id     49084 non-null  int64         
 1   category       49084 non-null  object        
 2   model_name     49084 non-null  object        
 3   sale_date      49084 non-null  datetime64[ns]
 4   quantity       49084 non-null  int64         
 5   revenue        49084 non-null  float64       
 6   stock_on_hand  49084 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(2)
memory usage: 3.0+ MB


In [18]:
print(f"Данные: {len(df)} строк, {df.columns.tolist()}")

Данные: 49084 строк, ['product_id', 'category', 'model_name', 'sale_date', 'quantity', 'revenue', 'stock_on_hand']


In [23]:
# Вариант Б: Запрос к БД 
# from src.db.queries import run_query ⬆
# QUERY = "SELECT ... -- запрос"
# df = run_query(QUERY) 

query = '''
SELECT 
    p.product_id,
    p.category,
    p.model_name,
    p.cost_price,
    p.sale_price,
    s.sale_id,
    s.sale_date::DATE AS sale_date,          -- Приводим к дате для удобства
    s.sale_date::TIME AS sale_time,          -- Выделяем время отдельно
    s.quantity,
    (s.quantity * p.sale_price) AS revenue,  -- Выручка по строке
    s.customer_id,
    inv.stock_on_hand,
    inv.operation_type
FROM sales s
JOIN products p ON s.product_id = p.product_id
LEFT JOIN inventory_log inv 
    ON s.product_id = inv.product_id 
    AND s.sale_date::DATE = inv.date
ORDER BY s.sale_date;
'''
df_2 = run_query(query)

# Сортируем от новых к старым
df_sorted = df.sort_values('sale_date', ascending=False)

# Проверяем результат
display(df_sorted.head(10))

2026-03-10 22:00:46.261 | INFO     | src.db.queries:run_query:28 - Query returned 49084 rows


,product_id,category,model_name,sale_date,quantity,revenue,stock_on_hand
49083,5,Clothing,Model_5,2026-03-07,3,97.50,47
49031,3,Clothing,Model_3,2026-03-06,6,640.44,110
49082,7,Clothing,Model_7,2026-03-06,3,181.35,109
49081,15,Clothing,Model_15,2026-03-06,4,234.80,21
49080,13,Clothing,Model_13,2026-03-06,4,216.56,25
49079,5,Clothing,Model_5,2026-03-06,4,130.00,23
49078,9,Clothing,Model_9,2026-03-06,4,133.60,103
49077,9,Clothing,Model_9,2026-03-06,2,66.80,103
49076,13,Clothing,Model_13,2026-03-06,6,324.84,25
49075,16,Electronics,Model_16,2026-03-06,6,265.08,46


In [24]:
df_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49084 entries, 0 to 49083
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      49084 non-null  int64  
 1   category        49084 non-null  object 
 2   model_name      49084 non-null  object 
 3   cost_price      49084 non-null  float64
 4   sale_price      49084 non-null  float64
 5   sale_id         49084 non-null  int64  
 6   sale_date       49084 non-null  object 
 7   sale_time       49084 non-null  object 
 8   quantity        49084 non-null  int64  
 9   revenue         49084 non-null  float64
 10  customer_id     49084 non-null  int64  
 11  stock_on_hand   49084 non-null  int64  
 12  operation_type  49084 non-null  object 
dtypes: float64(3), int64(5), object(5)
memory usage: 4.9+ MB


In [ ]:
# ЭКСПОРТ РЕЗУЛЬТАТОВ

# Сохранить очищенные данные
EXPORT_PATH = PROJECT_ROOT / "data" / "exports" / "00002_analysis_ready.csv"
EXPORT_PATH.parent.mkdir(exist_ok=True)
df.to_csv(EXPORT_PATH, index=False)
print(f"Экспорт: {EXPORT_PATH}")